# Skills — Agent-Analyse

Tier-1 eval (#386). Tests context-skill activation via `activate_skill`,
plus direct SkillManager prompt-delta diagnostic. Scoring via `expected.skills:`.

## 1. Setup

In [ ]:
import os, sys, importlib
from pathlib import Path

PYTF_ROOT = Path.cwd()
if PYTF_ROOT.name == 'notebooks':
    PYTF_ROOT = PYTF_ROOT.parent
os.chdir(PYTF_ROOT)
for p in [str(PYTF_ROOT / 'src'), str(PYTF_ROOT / 'notebooks')]:
    if p not in sys.path:
        sys.path.insert(0, p)

try:
    from dotenv import load_dotenv
    load_dotenv(PYTF_ROOT / '.env')
except ImportError:
    pass

try:
    import nest_asyncio; nest_asyncio.apply()
except ImportError:
    pass

import logging, structlog
logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(name)s | %(message)s')
structlog.configure(wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING))

import analysis_lib
importlib.reload(analysis_lib)
import analysis_lib as alib
print('analysis_lib OK -', len(alib.list_evaluators()), 'evaluators')

In [ ]:
from taskforce.application.factory import AgentFactory
from taskforce.application.executor import AgentExecutor

factory = AgentFactory()
executor = AgentExecutor(factory=factory)
alib.disable_post_mission_learning(executor)
print('factory + executor ready')

## 2. Agent bauen (mit Skill-Support)

Tools: `activate_skill`, `file_read`, `python`. Plus direkte SkillManager-Tests ohne Agent. Bundled Skills: pdf-processing, code-review, data-analysis.

In [ ]:
WORKDIR = Path('.taskforce_skills_analysis')
WORKDIR.mkdir(exist_ok=True)
SKILL_TOOLS = ['activate_skill', 'file_read', 'python']
SKILL_PROMPT = (
    'Du bist ein Assistent mit Skill-Support. Wenn ein User-Task einer '
    'spezialisierten Skill entspricht (z.B. PDF-Verarbeitung, Code-Review), '
    'aktiviere die passende Skill via activate_skill. Bei trivialen Tasks '
    '(Mathe, einfache Fragen) KEINE Skill aktivieren.'
)

async def build_skill_agent():
    a = await factory.create_agent(
        system_prompt=SKILL_PROMPT, tools=SKILL_TOOLS,
        persistence={'type': 'file', 'work_dir': str(WORKDIR)},
        work_dir=str(WORKDIR),
        planning_strategy='native_react', max_steps=6,
    )
    alib.patch_anti_compression(a)
    return a, len(SKILL_PROMPT)

BUILD_AGENT = build_skill_agent
_smoke, _chars = await BUILD_AGENT()
AVAILABLE_TOOLS = set(_smoke.tools.keys())
print(f'tools={sorted(AVAILABLE_TOOLS)}  prompt={_chars} chars')
await _smoke.close()

### 2a. Direkter SkillManager-Test (ohne Agent)

Misst den system-prompt-Delta wenn eine context-Skill aktiviert wird —
unabhängig vom Agent-Verhalten.

In [ ]:
from taskforce.infrastructure.skills.skill_loader import SkillLoader

skills_root = PYTF_ROOT / 'src' / 'taskforce' / 'skills'
loader = SkillLoader(skill_directories=[str(skills_root)])
found = list(loader.load_all_skills())
print(f'discovered {len(found)} bundled skills:')
for s in found[:12]:
    print(f'  - {s.name:30s} type={s.skill_type.value:8s}  desc={s.description[:50]}')

## 3. Szenarien laden

Aus `scenarios/skills.yaml`.

In [ ]:
all_scenarios = alib.load_scenarios('notebooks/scenarios/skills.yaml')
eligible = alib.filter_scenarios(all_scenarios, AVAILABLE_TOOLS)
print(f'Total: {len(all_scenarios)}, Eligible: {len(eligible)}')
for s in eligible:
    print(f'  - {s.id:35s} ({s.category}/{s.difficulty})')

## 4. Einzelnes Szenario (Detail-Lauf)

In [ ]:
TARGET_ID = 'skills-activate-pdf-processing'  # change me
target = next((s for s in eligible if s.id == TARGET_ID), None) or eligible[0]
print(f'Target: {target.id}')
print(f'Mission: {target.mission}')
print(f'Hidden intent: {target.hidden_intent}')

In [ ]:
agent, sys_chars = await BUILD_AGENT()
rec = await alib.run(
    executor, agent, target.mission,
    project_root=WORKDIR, snapshot_subdirs=(),
    initial_system_prompt_chars=sys_chars, silent=False,
)
# Stash MCP tool names if any (no-op for non-MCP notebooks)
rec.extra['mcp_tool_names'] = list(_mcp_tool_names_of(agent)) if '_mcp_tool_names_of' in dir() else []
alib.print_summary(rec)

## 5. Reports

In [ ]:
alib.print_tool_stats(rec)
print()
alib.print_tool_results(rec, head=15)

In [ ]:
card = alib.score_rule_based(rec, target)
print(f'=== Rule-based ({"PASS" if card.passed else "FAIL"}) ===')
alib.print_feature_checks(card)
if card.details:
    print('Details:')
    for d in card.details:
        print(f'  - {d}')

In [ ]:
await agent.close()
_ = alib.plot_tool_frequencies(rec, title=f'Tool-Aufrufe: {target.id}')

## 6. Batch-Lauf

In [ ]:
results = await alib.run_scenarios(
    executor, BUILD_AGENT, eligible,
    project_root=WORKDIR, snapshot_subdirs=(),
    reset_dirs_before_each=(),
    repeats=1, progress=True,
)  
print()
alib.print_scenario_summary(results)

In [ ]:
_ = alib.plot_scenario_matrix(results, metric='passed', title='Pass/Fail')
_ = alib.plot_scenario_matrix(results, metric='tool_calls', title='Tool calls')

## Ideen für weitere Experimente

Siehe TODOs in der Scenario-YAML.

In [ ]:
print('done')